# Submit de semillas individuales — Experimento 9101

Script standalone que aplica el corte 2000 a las 5 predicciones individuales
del semillerío (ya escritas en `./semillas/` por el notebook principal) y las
sube a Kaggle. **No re-entrena nada.**

Se corre en una sesión aparte, en el mismo directorio del experimento 9101
(donde existe la carpeta `semillas/` con los 5 archivos).

In [ ]:
# --- Setup ---
require("data.table")

# IMPORTANTE: ajustá el path al directorio donde el experimento 9101 dejó
# la carpeta 'semillas/' y la carpeta 'kaggle/'. Suele ser algo como:
#   ~/buckets/b1/exp/KA9101/
# o el path de trabajo del notebook original. Si ya estás en ese directorio,
# no hace falta setwd().
# setwd("~/buckets/b1/exp/KA9101")

PARAM <- list()
PARAM$experimento <- 9101
PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$corte <- 2000  # corte donde el ensemble dio su máximo Public
PARAM$semillas <- c(804043, 653561, 703903, 439693, 665857)

# chequeo previo: existen los 5 archivos?
for (s in PARAM$semillas) {
  arch <- paste0("semillas/prediccion_semilla_", s, ".txt")
  if (!file.exists(arch)) stop("No encontrado: ", arch)
}
cat("Los 5 archivos individuales están en disco. Listo para submit.\n")

dir.create("kaggle", showWarnings = FALSE)

In [ ]:
# --- (Opcional) Análisis local de solapamiento entre semillas ---
# Antes de subir a Kaggle, veamos qué tan distintas son las 5 predicciones
# en el corte 2000. Si las 5 marcan casi los mismos clientes, el semillerío
# aporta poca diversidad y los Public Scores van a ser muy parecidos entre sí.

top_por_semilla <- list()

for (s in PARAM$semillas) {
  tb <- fread(paste0("semillas/prediccion_semilla_", s, ".txt"))
  setorder(tb, -prob)
  top_por_semilla[[as.character(s)]] <- tb[1:PARAM$kaggle$corte, numero_de_cliente]
}

# clientes que aparecen en TODAS las semillas (intersección)
en_todas <- Reduce(intersect, top_por_semilla)

# clientes que aparecen en al menos UNA (unión)
en_alguna <- Reduce(union, top_por_semilla)

cat("Corte:", PARAM$kaggle$corte, "clientes por semilla\n")
cat("En las 5 semillas (intersección):", length(en_todas), "\n")
cat("En al menos 1 semilla (unión):    ", length(en_alguna), "\n")
cat("Estabilidad (intersección / corte):",
    round(length(en_todas) / PARAM$kaggle$corte, 3), "\n")

# Interpretación:
# - Estabilidad > 0.90 → semillas muy correlacionadas, ensemble aporta poco
# - Estabilidad 0.75-0.90 → diversidad moderada, ensemble típico
# - Estabilidad < 0.75 → mucha diversidad, ensemble debería ayudar bastante

In [ ]:
# --- Submit de las 5 semillas individuales a Kaggle ---

for (semilla in PARAM$semillas) {

  tb_ind <- fread(paste0("semillas/prediccion_semilla_", semilla, ".txt"))
  setorder(tb_ind, -prob)
  tb_ind[, Predicted := 0L]
  tb_ind[1:PARAM$kaggle$corte, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento,
                           "_semilla_", semilla,
                           "_", PARAM$kaggle$corte, ".csv")

  fwrite(tb_ind[, list(numero_de_cliente, Predicted)],
         file = archivo_kaggle, sep = ",")

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'semilla_individual=", semilla,
                    " envios=", PARAM$kaggle$corte, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  cat(format(Sys.time(), "%X"), " - submit semilla ", semilla, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nListo. Revisá los 5 Public Scores en la solapa Submissions de Kaggle.\n")